# Module 8: Governance, Safety & Compliance Engineering

Build a `SecurityMiddleware` shield for prompt injection defense and PII-safe logging.


## 1. Setup

Load middleware utilities and sample notes.


In [ ]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from security_middleware import SecurityMiddleware, sample_receipt_notes

print("✓ Setup complete")


## 2. Initialize Security Middleware


In [ ]:
middleware = SecurityMiddleware()
notes = sample_receipt_notes()
notes


## 3. Threat Modeling Tests

Detect prompt injection and jailbreak patterns before LLM calls.


In [ ]:
scan_results = {}
for name, text in notes.items():
    scan_results[name] = middleware.scan_input(text).to_dict()

scan_results


## 4. PII Redaction

Sanitize sensitive fields before logging outputs.


In [ ]:
text_with_pii = "Card 4111 1111 1111 1111, email jane.doe@corp.com, phone 415-555-0199, ssn 123-45-6789"
redacted = middleware.redact_pii(text_with_pii)

print("Original:", text_with_pii)
print("Redacted:", redacted)


In [ ]:
llm_output = {
    "decision": "denied",
    "reason": "Card 4111 1111 1111 1111 appears in receipt image for jane.doe@corp.com",
}
sanitized_output = middleware.sanitize_output_for_logging(llm_output)
sanitized_output


## 5. Security Event Log

Every scan/sanitization action is logged for governance auditability.


In [ ]:
middleware.security_log


## 6. Enforcement Policy Demo

Block medium/high risk inputs from reaching the model.


In [ ]:
def guarded_llm_call(user_note: str):
    result = middleware.scan_input(user_note)
    if not result.allowed:
        return {
            "blocked": True,
            "risk_level": result.risk_level,
            "reasons": result.reasons,
            "message": "Request blocked by SecurityMiddleware",
        }
    return {
        "blocked": False,
        "risk_level": result.risk_level,
        "sanitized_prompt": result.sanitized_text,
        "message": "Allowed to proceed",
    }

guarded_llm_call(notes["injection"]), guarded_llm_call(notes["safe"])


## 7. Assertions (Smoke Checks)


In [ ]:
assert scan_results["safe"]["allowed"] is True
assert scan_results["injection"]["allowed"] is False
assert scan_results["jailbreak"]["allowed"] is False
assert "[REDACTED_CREDIT_CARD]" in redacted
assert "[REDACTED_EMAIL]" in sanitized_output["reason"]

print("✓ Module 8 governance/safety smoke checks passed")
